In [36]:
import os
os.getcwd()

'/Users/subhadeepdutta/Documents/GitHub/algorithmic-trading/src'

In [37]:
os.listdir("../data")

['.DS_Store', 'README.md', 'clean', 'raw']

In [38]:
os.listdir("../data/clean")

['clean_prices.parquet', '.ipynb_checkpoints', 'features.parquet']

In [69]:
import pyarrow.parquet as pq
path = "../data/clean/clean_prices.parquet"
table = pq.read_table(path, use_pandas_metadata=False)
wide = table.to_pandas(ignore_metadata=True)
df = table.to_pandas(ignore_metadata=True)
df.head()

,Date,"('Close', 'AAPL')","('Close', 'AMZN')","('Close', 'IBM')","('Close', 'NOW')","('Close', 'NVDA')","('Close', 'PLUG')","('Close', 'QUBT')","('Close', 'RHM.DE')","('Close', 'SPY')",...,"('Volume', 'AAPL')","('Volume', 'AMZN')","('Volume', 'IBM')","('Volume', 'NOW')","('Volume', 'NVDA')","('Volume', 'PLUG')","('Volume', 'QUBT')","('Volume', 'RHM.DE')","('Volume', 'SPY')","('Volume', 'TSLA')"
0,2015-01-02,24.237551,15.4260,98.123062,13.476,0.483011,2.98,0.2,29.793606,170.589615,...,212818400.0,55664000.0,5779673.0,4079000.0,113680000.0,3957700.0,0.0,148471.0,121465900.0,71466000.0
1,2015-01-05,23.554743,15.1095,96.579086,13.430,0.474853,3.01,0.2,28.750767,167.508804,...,257142000.0,55484000.0,5104898.0,4002500.0,197952000.0,6713900.0,1.0,269397.0,169632600.0,80527500.0
2,2015-01-06,23.556953,14.7645,94.496262,13.266,0.460457,3.05,0.2,28.829706,165.931015,...,263188400.0,70380000.0,6429448.0,4279500.0,197764000.0,5997600.0,0.0,189323.0,209151400.0,93928500.0
3,2015-01-07,23.887272,14.9210,93.878700,13.418,0.459257,3.05,0.2,28.750767,167.998703,...,160423600.0,52806000.0,4918083.0,3115000.0,321808000.0,3007400.0,0.0,165350.0,125346700.0,44526000.0
4,2015-01-08,24.805079,15.0230,95.919128,13.778,0.476533,3.08,0.2,29.843462,170.979919,...,237458000.0,61768000.0,4431693.0,3185500.0,283780000.0,3203100.0,300.0,151718.0,147217800.0,51637500.0


In [89]:
df = close.melt(id_vars="date", var_name="ticker", value_name="close")
df = df.sort_values(["ticker", "date"])
df.head()


,date,ticker,close
0,2015-01-02,"('', 'AAPL')",24.237551
1,2015-01-05,"('', 'AAPL')",23.554743
2,2015-01-06,"('', 'AAPL')",23.556953
3,2015-01-07,"('', 'AAPL')",23.887272
4,2015-01-08,"('', 'AAPL')",24.805079


In [90]:
import numpy as np
df["ticker"] = df["ticker"].astype(str).str.replace(r"[()'\",]", "", regex=True)
df["daily_return"] = df.groupby("ticker")["close"].pct_change()
df["log_return"] = np.log1p(df["daily_return"])
df["ma_20"] = df.groupby("ticker")["close"].transform(lambda s: s.rolling(20).mean())
df["vol_20"] = df.groupby("ticker")["daily_return"].transform(lambda s: s.rolling(20).std())

features = df.dropna(subset=["daily_return", "log_return", "ma_20", "vol_20"]).copy()
features.to_parquet("../data/clean/features.parquet", index=False)

features.head()


,date,ticker,close,daily_return,log_return,ma_20,vol_20
20,2015-02-02,AAPL,26.299282,0.012547,0.012469,24.631386,0.023813
21,2015-02-03,AAPL,26.303715,0.000169,0.000169,24.768835,0.022587
22,2015-02-04,AAPL,26.505451,0.007670,0.007640,24.916260,0.022551
23,2015-02-05,AAPL,26.694641,0.007138,0.007112,25.056628,0.022477
24,2015-02-06,AAPL,26.469847,-0.008421,-0.008457,25.139867,0.021310
